# Compare experiment metrics

Load aggregate metrics JSON files from one folder and compare each experiment in a single table. By default, this reads only `metrics_all_scenarios*.json` files in `anomaly_detection/experiment_outputs` (not the per-scenario metric files). Change `METRICS_DIR` or `METRICS_PATTERN` below to compare another folder or filename pattern.

In [10]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def find_project_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiment_outputs").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
        nested_project = candidate / "anomaly_detection"
        if (nested_project / "experiment_outputs").is_dir() and (nested_project / "pyproject.toml").is_file():
            return nested_project
    raise FileNotFoundError(
        "Could not find the anomaly_detection project directory. "
        "Run this notebook from within that project or one of its subfolders."
    )


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
METRICS_DIR = PROJECT_DIR / "experiment_outputs"
METRICS_PATTERN = "metrics_all_scenarios*.json"
REQUIRED_FIELDS = (
    "runs",
    "TP_total",
    "FP_total",
    "FN_total",
    "precision_global",
    "recall_global",
    "f1_global",
)

metrics_files = sorted(METRICS_DIR.glob(METRICS_PATTERN))
if not metrics_files:
    raise FileNotFoundError(
        f"No files matching {METRICS_PATTERN!r} were found in {METRICS_DIR}"
    )

rows = []
for path in metrics_files:
    with path.open(encoding="utf-8") as metrics_file:
        metrics = json.load(metrics_file)
    missing_fields = set(REQUIRED_FIELDS).difference(metrics)
    if missing_fields:
        raise ValueError(f"{path.name} is missing fields: {sorted(missing_fields)}")

    experiment_name = path.stem.removeprefix("metrics_all_scenarios").lstrip("_")
    rows.append({
        "experiment": experiment_name or "default",
        **{field: metrics[field] for field in REQUIRED_FIELDS},
    })

results = pd.DataFrame(rows).sort_values(
    ["f1_global", "experiment"], ascending=[False, True]
).reset_index(drop=True)
display(results.round({
    "precision_global": 3,
    "recall_global": 3,
    "f1_global": 3,
}))
print(f"Compared {len(results)} experiment files from {METRICS_DIR}")

,experiment,runs,TP_total,FP_total,FN_total,precision_global,recall_global,f1_global
0,v2.0.0,29,12,3,17,0.800,0.414,0.545
1,more_alerts_48h_0.7_uptime_any,29,11,6,18,0.647,0.379,0.478
2,more_alerts_24h_0.7_uptime_any,29,12,12,17,0.500,0.414,0.453
3,0.7_uptime_any,29,9,2,20,0.818,0.310,0.450
4,0.7,29,9,3,20,0.750,0.310,0.439
5,0.75,29,9,3,20,0.750,0.310,0.439
6,0.8,29,9,3,20,0.750,0.310,0.439
7,more_alerts_24h_0.7_uptime_any_old,29,8,3,21,0.727,0.276,0.400
8,more_alerts_24h_0.7_uptime_any_>=old,29,10,11,19,0.476,0.345,0.400
9,more_alerts_12h_0.7_uptime_any,29,11,18,18,0.379,0.379,0.379


Compared 38 experiment files from /mnt/c/Users/Kevin Santos/Documents/Programming/tractian_challenge/anomaly_detection/experiment_outputs
